# ADS 509 Module 1: APIs and Web Scraping

This notebook has two parts. In the first part, you will scrape lyrics from AZLyrics.com. In the second part, you'll run code that verifies the completeness of your data pull. 

For this assignment you have chosen two musical artists who have at least 20 songs with lyrics on AZLyrics.com. We start with pulling some information and analyzing them.


## General Assignment Instructions

These instructions are included in every assignment, to remind you of the coding standards for the class. Feel free to delete this cell after reading it. 

One sign of mature code is conforming to a style guide. We recommend the [Google Python Style Guide](https://google.github.io/styleguide/pyguide.html). If you use a different style guide, please include a cell with a link. 

Your code should be relatively easy-to-read, sensibly commented, and clean. Writing code is a messy process, so please be sure to edit your final submission. Remove any cells that are not needed or parts of cells that contain unnecessary code. Remove inessential `import` statements and make sure that all such statements are moved into the designated cell. 

Make use of non-code cells for written commentary. These cells should be grammatical and clearly written. In some of these cells you will have questions to answer. The questions will be marked by a "Q:" and will have a corresponding "A:" spot for you. *Make sure to answer every question marked with a `Q:` for full credit.* 


# Importing Libraries

In [1]:
import os
import datetime
import re

# for the lyrics scrape section
import requests
import time
from bs4 import BeautifulSoup
from collections import defaultdict, Counter
import random

In [2]:
# Use this cell for any import statements you add
import shutil

---

# Lyrics Scrape

This section asks you to pull data by scraping www.AZLyrics.com. In the notebooks where you do that work you are asked to store the data in specific ways. 

In [3]:
artists = {'eagles':"https://www.azlyrics.com/e/eagles.html",
           'journey':"https://www.azlyrics.com/j/journey.html"} 
# we'll use this dictionary to hold both the artist name and the link on AZlyrics

## A Note on Rate Limiting

The lyrics site, www.azlyrics.com, does not have an explicit maximum on number of requests in any one time, but in our testing it appears that too many requests in too short a time will cause the site to stop returning lyrics pages. (Entertainingly, the page that gets returned seems to only have the song title to [a Tom Jones song](https://www.azlyrics.com/lyrics/tomjones/itsnotunusual.html).) 

Whenever you call `requests.get` to retrieve a page, put a `time.sleep(5 + 10*random.random())` on the next line. This will help you not to get blocked. If you _do_ get blocked, which you can identify if the returned pages are not correct, just request a lyrics page through your browser. You'll be asked to perform a CAPTCHA and then your requests should start working again. 

## Part 1: Finding Links to Songs Lyrics

That general artist page has a list of all songs for that artist with links to the individual song pages. 

Q: Take a look at the `robots.txt` page on www.azlyrics.com. (You can read more about these pages [here](https://developers.google.com/search/docs/advanced/robots/intro).) Is the scraping we are about to do allowed or disallowed by this page? How do you know? 

A: Scraping is allowed on this page. The web page allows pulling data by artist or song title.

In [4]:
# Let's set up a dictionary of lists to hold our links
lyrics_pages = defaultdict(list)

for artist, artist_page in artists.items() :
    # request the page and sleep
    r = requests.get(artist_page)
    time.sleep(5 + 10*random.random())

    # now extract the links to lyrics pages from this page
    if r.status_code == 200:
        soup = BeautifulSoup(r.text, "html.parser")
    
    # store the links `lyrics_pages` where the key is the artist and the
    # value is a list of links
    for page_link in soup.find_all('a',href=re.compile("/lyrics/")) :
        lyrics_pages[artist].append(page_link.attrs['href'])

Let's make sure we have enough lyrics pages to scrape. 

In [5]:
for artist, lp in lyrics_pages.items() :
    assert(len(set(lp)) > 20) 

In [6]:
# Let's see how long it's going to take to pull these lyrics 
# if we're waiting `5 + 10*random.random()` seconds 
for artist, links in lyrics_pages.items() : 
    print(f"For {artist} we have {len(links)}.")
    print(f"The full pull will take for this artist will take {round(len(links)*10/3600,2)} hours.")

For eagles we have 103.
The full pull will take for this artist will take 0.29 hours.
For journey we have 188.
The full pull will take for this artist will take 0.52 hours.


## Part 2: Pulling Lyrics

Now that we have the links to our lyrics pages, let's go scrape them! Here are the steps for this part. 

1. Create an empty folder in our repo called "lyrics". 
1. Iterate over the artists in `lyrics_pages`. 
1. Create a subfolder in lyrics with the artist's name. For instance, if the artist was Cher you'd have `lyrics/cher/` in your repo.
1. Iterate over the pages. 
1. Request the page and extract the lyrics from the returned HTML file using BeautifulSoup.
1. Use the function below, `generate_filename_from_url`, to create a filename based on the lyrics page, then write the lyrics to a text file with that name. 


In [7]:
def generate_filename_from_link(link) :
    
    if not link :
        return None
    
    # drop the http or https and the html
    name = link.replace("https","").replace("http","")
    name = link.replace(".html","")

    name = name.replace("/lyrics/","")
    
    # Replace useless chareacters with UNDERSCORE
    name = name.replace("://","").replace(".","_").replace("/","_")
    
    # tack on .txt
    name = name + ".txt"
    
    return(name)


In [8]:
# Make the lyrics folder here. If you'd like to practice your programming, add functionality 
# that checks to see if the folder exists. If it does, then use shutil.rmtree to remove it and create a new one.

if os.path.isdir("lyrics") : 
    shutil.rmtree("lyrics/")

os.mkdir("lyrics")

In [10]:
url_stub = "https://www.azlyrics.com"
start = time.time()

total_pages = 0

for artist in lyrics_pages:
    
    # 1. Build a subfolder for the artist
    os.mkdir("lyrics/" + artist)
    pages = 0

    # 2. Iterate over the lyrics pages
    for page in lyrics_pages[artist]:
        
        page_name = page.replace("/lyrics/", "").replace(".html", "")
        print(f"Requesting: {page_name}")
        
        # 3. Request the lyrics page.
        url = url_stub + page
        
        # Make the request
        r = requests.get(url)
        
        # Don't forget to add a line like `time.sleep(5 + 10*random.random())`
        # to sleep after making the request
        time.sleep(5 + 10 * random.random())

        pages += 1

        if r.status_code != 200:
            print(f"Got a bad status code ({r.status_code}) on {page}.")
        else:
            # 4. Extract the title and lyrics from the page
            soup = BeautifulSoup(r.text, "html.parser")
            output_filename = "lyrics/" + artist + "/" + generate_filename_from_link(page)

            # First let's get the song title
            for item in soup.find_all("b"):
                if '"' in item.text:
                    title = item.text
                    break

            # Let's get the lyrics
            hit_ringtone = False
            for item in soup.find_all("div"):
                # Lyrics are the div after the ringtone
                if hit_ringtone:
                    break

                if "class" in item.attrs:
                    if "ringtone" in item.attrs["class"]:
                        hit_ringtone = True

                lyrics = item.text

            # Special case: Delay on certain known pages
            if title == '"It\'s Not Unusual"':
                print("Oops, looks like we went too fast. Getting Tom Jones... back.")
                time.sleep(360 + random.random() * 80)  # see if a long sleep works

            # 5. Write out the title, two returns ('\n\n'), and the lyrics
            with open(output_filename, 'w') as ofile:
                ofile.write(title + "\n\n")
                ofile.write(lyrics)

        # Optional limit:
        # if pages == 10:
        #     break

    # Sleep between artists
    time.sleep(60)
    total_pages += pages

Requesting: eagles/takeiteasy-1972
Requesting: eagles/witchywoman
Requesting: eagles/chugallnight
Requesting: eagles/mostofusaresad
Requesting: eagles/nightingale
Requesting: eagles/trainleavesherethismorning
Requesting: eagles/takethedevil
Requesting: eagles/earlybird
Requesting: eagles/peacefuleasyfeeling
Requesting: eagles/tryin
Requesting: eagles/doolindalton
Requesting: eagles/twentyone
Requesting: eagles/outofcontrol
Requesting: eagles/tequilasunrise19388
Requesting: eagles/desperado
Requesting: eagles/certainkindoffool
Requesting: eagles/outlawman
Requesting: eagles/saturdaynight
Requesting: eagles/bittercreek
Requesting: eagles/doolindaltondesperadoreprise
Requesting: eagles/alreadygone
Requesting: eagles/younevercrylikealover
Requesting: eagles/midnightflyer
Requesting: eagles/myman
Requesting: eagles/ontheborder
Requesting: eagles/jamesdean
Requesting: eagles/ol55
Requesting: eagles/isittrue
Requesting: eagles/gooddayinhell
Requesting: eagles/thebestofmylove
Requesting: eagle

In [11]:
print(f"Total run time was {round((time.time() - start)/3600,2)} hours.")

Total run time was 0.97 hours.


---

# Evaluation

This assignment asks you to pull data by scraping www.AZLyrics.com.  After you have finished the above sections , run all the cells in this notebook. Print this to PDF and submit it, per the instructions.

In [12]:
# Simple word extractor from Peter Norvig: https://norvig.com/spell-correct.html
def words(text): 
    return re.findall(r'\w+', text.lower())

## Checking Lyrics 

The output from your lyrics scrape should be stored in files located in this path from the directory:
`/lyrics/[Artist Name]/[filename from URL]`. This code summarizes the information at a high level to help the instructor evaluate your work. 

In [13]:
artist_folders = os.listdir("lyrics/")
artist_folders = [f for f in artist_folders if os.path.isdir("lyrics/" + f)]

for artist in artist_folders : 
    artist_files = os.listdir("lyrics/" + artist)
    artist_files = [f for f in artist_files if 'txt' in f or 'csv' in f or 'tsv' in f]

    print(f"For {artist} we have {len(artist_files)} files.")

    artist_words = []

    for f_name in artist_files : 
        with open("lyrics/" + artist + "/" + f_name) as infile : 
            artist_words.extend(words(infile.read()))

            
    print(f"For {artist} we have roughly {len(artist_words)} words, {len(set(artist_words))} are unique.")


For eagles we have 103 files.
For eagles we have roughly 369 words, 201 are unique.
For journey we have 187 files.
For journey we have roughly 606 words, 318 are unique.
